In [1]:
import pandas as pd

# Sample review dataset
data = {
    'review': [
        "This product is absolutely amazing! I love how well it works.",
        "Terrible quality, broke on the first day. Very disappointed.",
        "Great customer service and fast shipping. Highly recommended!",
        "Worst experience ever. The item arrived damaged and unusable.",
        "Decent value for the price. Works as expected.",
        "Extremely satisfied with this purchase! Excellent build quality."
    ],
    'sentiment': ['positive', 'negative', 'positive', 'negative', 'positive', 'positive']
}

df = pd.DataFrame(data)

# Map sentiment labels to binary integers (1 = Positive, 0 = Negative)
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print("Dataset Preview:")
print(df[['review', 'label']].head())

Dataset Preview:
                                              review  label
0  This product is absolutely amazing! I love how...      1
1  Terrible quality, broke on the first day. Very...      0
2  Great customer service and fast shipping. High...      1
3  Worst experience ever. The item arrived damage...      0
4     Decent value for the price. Works as expected.      1


In [2]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK corpora
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # 1. Lowercase the text
    text = text.lower()

    # 2. Remove HTML tags and non-alphabetical characters (punctuation/numbers)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)

    # 3. Tokenize words by splitting on whitespace
    tokens = text.split()

    # 4. Remove stopwords and lemmatize remaining tokens to root form
    cleaned_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    # 5. Rejoin into a clean string
    return " ".join(cleaned_tokens)

# Apply cleaning function to reviews
df['cleaned_review'] = df['review'].apply(clean_text)

print("Original vs Cleaned Sample:")
print("Before:", df['review'].iloc[0])
print("After: ", df['cleaned_review'].iloc[0])

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Original vs Cleaned Sample:
Before: This product is absolutely amazing! I love how well it works.
After:  product absolutely amazing love well work


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Define Features (X) and Target (y)
X = df['cleaned_review']
y = df['label']

# 2. Stratified Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Initialize TF-IDF Vectorizer
# max_features limits vocabulary size to top 5000 terms
tfidf = TfidfVectorizer(max_features=5000)

# Fit vectorizer ON TRAINING DATA ONLY, then transform both sets
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Vocabulary Size:", len(tfidf.vocabulary_))
print("TF-IDF Matrix Shape (Train):", X_train_tfidf.shape)

Vocabulary Size: 24
TF-IDF Matrix Shape (Train): (4, 24)


In [4]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Initialize classifiers
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(random_state=42)
}

# 2. Train and evaluate each model
for name, model in models.items():
    # Fit model on transformed TF-IDF training vectors
    model.fit(X_train_tfidf, y_train)

    # Predict on test vectors
    y_pred = model.predict(X_test_tfidf)

    print(f"\n================ {name} ================")
    print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))


================ Multinomial Naive Bayes ================
Accuracy Score: 0.5000
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00         1
    Positive       0.50      1.00      0.67         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2


================ Logistic Regression ================
Accuracy Score: 0.5000
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00         1
    Positive       0.50      1.00      0.67         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/m

In [8]:
# =========================================================
# Step 5 (Updated): Out-of-Vocabulary & Neutral-Aware Inference
# =========================================================

def predict_sentiment_robust(raw_review_text, model, vectorizer, confidence_threshold=0.60):
    """
    Predicts sentiment while checking for out-of-vocabulary (OOV)
    words and low-confidence predictions to return 'Neutral/Unknown'.
    """
    # 1. Preprocess raw text using our cleaning pipeline
    cleaned = clean_text(raw_review_text)

    # 2. Vectorize using the pre-fitted TF-IDF instance
    vectorized_input = vectorizer.transform([cleaned])

    # 3. Check if any input words match the trained vocabulary (nnz = number of non-zero elements)
    if vectorized_input.nnz == 0:
        return {
            "Raw Input": raw_review_text,
            "Cleaned Text": cleaned,
            "Predicted Sentiment": "Neutral / Unknown",
            "Confidence Score": "N/A (No recognized vocabulary words)"
        }

    # 4. Extract class prediction and probability distribution
    prediction = model.predict(vectorized_input)[0]
    probabilities = model.predict_proba(vectorized_input)[0]
    top_confidence = probabilities[prediction]

    # 5. Handle low-confidence decisions near 50-60% split
    if top_confidence < confidence_threshold:
        sentiment_label = "Neutral"
    else:
        sentiment_label = "Positive" if prediction == 1 else "Negative"

    return {
        "Raw Input": raw_review_text,
        "Cleaned Text": cleaned,
        "Predicted Sentiment": sentiment_label,
        "Confidence Score": f"{top_confidence * 100:.2f}%"
    }

# ---------------------------------------------------------
# Test with OOV and Out-of-Domain Inputs
# ---------------------------------------------------------
test_inputs = [
    "new text file for my nlp",  # Out of domain / OOV words
    "This product is absolutely amazing!",  # Strong Positive
    "Terrible quality, broke instantly."   # Strong Negative
]

print("--- Robust Sentiment Inference Results ---")
for text in test_inputs:
    res = predict_sentiment_robust(text, models["Multinomial Naive Bayes"], tfidf)
    print(f"\nInput: '{res['Raw Input']}'")
    print(f"Predicted Sentiment: {res['Predicted Sentiment']} | Confidence: {res['Confidence Score']}")

--- Robust Sentiment Inference Results ---

Input: 'new text file for my nlp'
Predicted Sentiment: Neutral / Unknown | Confidence: N/A (No recognized vocabulary words)

Input: 'This product is absolutely amazing!'
Predicted Sentiment: Positive | Confidence: 80.67%

Input: 'Terrible quality, broke instantly.'
Predicted Sentiment: Neutral / Unknown | Confidence: N/A (No recognized vocabulary words)
